# Agente Connect-4 — MCTS con UCB1
**Fundamentos de Inteligencia Artificial · Universidad de La Sabana · 2026-1**

**Autor:** Pablo [Apellido] · **Profesor:** Felix Mohr

---

Este notebook sigue la metodología **CRISP-DM** *(Cross Industry Standard Process for Data Mining)*,
el estándar de la industria para proyectos de analítica de datos, aplicada al diseño,
análisis y optimización de un agente MCTS para Connect-4.

| Fase CRISP-DM | Sección |
|---|---|
| Business Understanding | 1 |
| Data Understanding | 2 |
| Data Preparation | 3 |
| Modeling (Experimentos) | 4 |
| Evaluation | 5 |
| Deployment | 6 |


## 0. Instalación y Setup

In [ ]:
# Instalar dependencias necesarias
!pip install numpy pandas matplotlib seaborn scikit-learn pydantic --quiet


In [ ]:
# Subir el zip del torneo y descomprimirlo
from google.colab import files
uploaded = files.upload()   # selecciona tournament.zip


In [ ]:
import zipfile, os

with zipfile.ZipFile('tournament.zip', 'r') as z:
    z.extractall('.')

# Pararse en la carpeta del torneo
os.chdir('tournament')
print("Archivos disponibles:", os.listdir('.'))


In [ ]:
# Imports globales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 5)

print("✓ Imports listos")


In [ ]:
# Importar el entorno y el agente
import sys
sys.path.insert(0, '.')

from connect4.connect_state import ConnectState
from connect4.policy import Policy
from groups.GroupA.policy import PabloMCTS   # ← cambia GroupA por tu carpeta real

# Jugador aleatorio (baseline obligatorio de la rúbrica)
class RandomAgent(Policy):
    def mount(self): pass
    def act(self, s: np.ndarray) -> int:
        free = [c for c in range(7) if s[0, c] == 0]
        return int(np.random.choice(free))

print("✓ Agentes listos")


In [ ]:
def play_match(agent, opponent, n_games=100, time_budget_ms=200, c=1.414):
    """
    Corre n_games partidas entre agent y opponent.
    Alterna quién va primero (50% como rojo, 50% como amarillo).
    Retorna un DataFrame con el log completo de todas las jugadas.
    """
    agent.time_budget_ms = time_budget_ms
    agent.c = c

    all_logs = []

    for game_i in range(n_games):
        agent.mount()
        opponent.mount()

        # Alternar color del agente
        if game_i % 2 == 0:
            players   = {-1: agent, 1: opponent}
            my_color  = -1
        else:
            players   = {-1: opponent, 1: agent}
            my_color  = 1

        state = ConnectState()
        while not state.is_final():
            action = players[state.player].act(state.board)
            state  = state.transition(int(action))

        winner = state.get_winner()
        if winner == my_color:
            result = 'win'
        elif winner == 0:
            result = 'draw'
        else:
            result = 'loss'

        # Etiquetar cada jugada con metadatos de la partida
        for move in agent.game_log:
            move['game_id']      = game_i
            move['game_result']  = result
            move['opponent']     = type(opponent).__name__
            all_logs.append(move)

    return pd.DataFrame(all_logs)

print("✓ Función play_match lista")


## 1. Business Understanding

El torneo asigna a cada agente un **presupuesto de tiempo tipo ajedrez**: el tiempo no es por partida
sino acumulado, y cada llamada a `act()` descuenta del total. Esto convierte el problema en una
pregunta de optimización concreta:

> **¿Cómo maximizar la calidad de decisión por milisegundo disponible?**

### Métricas de éxito
| Métrica | Objetivo mínimo (rúbrica) | Objetivo real |
|---|---|---|
| Win rate vs aleatorio | > 50% en ambos colores | > 90% |
| Win rate en self-play | Analizar ventaja de color | Cercano a 50% |
| Simulaciones/ms | Baseline de referencia | Maximizar |

### Variables de configuración del agente
- `time_budget_ms`: presupuesto de tiempo por jugada
- `c`: parámetro de exploración en UCB1 (controla exploración vs explotación)


## 2. Data Understanding

Cada llamada a `act()` registra las siguientes variables internas del agente.
Primero corremos una muestra pequeña para entender las distribuciones base.


In [ ]:
# Muestra exploratoria: 30 partidas con configuración base
agent_explore = PabloMCTS(time_budget_ms=200, c=1.414)
rand_explore  = RandomAgent()

df_explore = play_match(agent_explore, rand_explore, n_games=30, time_budget_ms=200, c=1.414)

print("Shape del DataFrame:", df_explore.shape)
print("\nColumnas:")
print(df_explore.dtypes)
print("\nPrimeras filas:")
df_explore.head()


In [ ]:
# Estadísticas descriptivas de las variables clave
df_explore[['simulations', 'tree_depth_max', 'root_confidence', 'time_used_ms']].describe().round(2)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

vars_plot = [
    ('simulations',    'Simulaciones por jugada'),
    ('tree_depth_max', 'Profundidad máxima del árbol'),
    ('root_confidence','Confianza del nodo raíz (Q/N)'),
    ('time_used_ms',   'Tiempo usado (ms)'),
]

for ax, (col, title) in zip(axes, vars_plot):
    sns.histplot(df_explore[col], ax=ax, kde=True, color='steelblue')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('')

plt.suptitle('Distribución de variables internas del agente — exploración inicial', y=1.02)
plt.tight_layout()
plt.savefig('fig_data_understanding.png', bbox_inches='tight')
plt.show()
print("✓ Figura guardada")


## 3. Data Preparation

Antes de los experimentos definimos el pipeline de limpieza que se aplicará
a **todos** los DataFrames generados. Esto cubre el indicador de preprocesamiento
de la materia de Analítica.


In [ ]:
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pipeline de limpieza y transformación aplicado a todos los logs.
    Pasos:
      1. Eliminar jugadas donde el agente no hizo ninguna simulación
         (edge case: tablero casi lleno, solo una columna libre)
      2. Normalizar tiempo usado sobre el presupuesto total
      3. Crear variable derivada 'efficiency': simulaciones por ms
      4. Codificar game_result como variable numérica para modelos
      5. Codificar color como variable legible
    """
    df = df.copy()

    # 1. Eliminar jugadas sin simulaciones reales
    before = len(df)
    df = df[df['simulations'] > 0].reset_index(drop=True)
    print(f"  Filas eliminadas (sim=0): {before - len(df)}")

    # 2. Tiempo normalizado (0-1 sobre el presupuesto)
    df['time_norm'] = df['time_used_ms'] / df['time_budget_ms']

    # 3. Eficiencia: simulaciones por milisegundo
    df['efficiency'] = df['simulations'] / df['time_used_ms'].replace(0, np.nan)

    # 4. Resultado numérico para modelos
    result_map = {'win': 1, 'draw': 0, 'loss': -1}
    df['result_num'] = df['game_result'].map(result_map)

    # 5. Color legible
    df['color_label'] = df['my_color'].map({-1: 'Rojo', 1: 'Amarillo'})

    return df

# Verificar con la muestra exploratoria
df_clean_sample = clean_df(df_explore)
print(f"Shape final: {df_clean_sample.shape}")
df_clean_sample[['simulations', 'time_norm', 'efficiency', 'result_num', 'color_label']].head()


## 4. Modeling — Experimentos

Cuatro experimentos que varían los parámetros de configuración del agente.
Cada uno genera su propio DataFrame limpio y sus gráficas.


### Experimento 1 — Variación del parámetro `c` de UCB1

**Pregunta:** ¿Cuánta exploración conviene en Connect-4?
El valor teórico es `c = √2 ≈ 1.414`, pero el óptimo empírico puede diferir.
Corremos 100 partidas contra el aleatorio para cada valor de `c`.


In [ ]:
C_VALUES    = [0.5, 0.8, 1.0, 1.414, 1.5, 2.0]
N_GAMES_EXP = 100
BUDGET_EXP1 = 200   # ms fijo para aislar el efecto de c

results_exp1 = []

for c_val in C_VALUES:
    print(f"  c = {c_val:.3f} ...", end=' ')
    agent = PabloMCTS(time_budget_ms=BUDGET_EXP1, c=c_val)
    rand  = RandomAgent()
    df    = play_match(agent, rand, n_games=N_GAMES_EXP,
                       time_budget_ms=BUDGET_EXP1, c=c_val)
    df    = clean_df(df)

    # Win rate por partida (una fila por partida)
    per_game = df.groupby('game_id')['result_num'].first()
    win_rate = (per_game == 1).mean()
    avg_sims = df['simulations'].mean()

    results_exp1.append({
        'c': c_val,
        'win_rate': win_rate,
        'avg_simulations': avg_sims,
    })
    print(f"win rate = {win_rate:.0%}")

df_exp1 = pd.DataFrame(results_exp1)
df_exp1


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Win rate vs c
axes[0].plot(df_exp1['c'], df_exp1['win_rate'], marker='o', color='steelblue', linewidth=2)
axes[0].axvline(x=1.414, linestyle='--', color='gray', alpha=0.6, label='√2 (teórico)')
best_c = df_exp1.loc[df_exp1['win_rate'].idxmax(), 'c']
axes[0].axvline(x=best_c, linestyle='--', color='tomato', alpha=0.8, label=f'Óptimo empírico (c={best_c})')
axes[0].set_xlabel('Parámetro c (exploración UCB1)')
axes[0].set_ylabel('Win rate vs jugador aleatorio')
axes[0].set_title('Win rate vs parámetro c de UCB1')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0].legend()

# Simulaciones promedio vs c
axes[1].bar(df_exp1['c'].astype(str), df_exp1['avg_simulations'], color='steelblue', alpha=0.8)
axes[1].set_xlabel('Parámetro c')
axes[1].set_ylabel('Simulaciones promedio por jugada')
axes[1].set_title('Simulaciones promedio vs parámetro c')

plt.suptitle(f'Experimento 1 — Variación de c | {N_GAMES_EXP} partidas vs aleatorio | {BUDGET_EXP1}ms/jugada')
plt.tight_layout()
plt.savefig('fig_exp1_c_param.png', bbox_inches='tight')
plt.show()

print(f"\n→ c óptimo empírico: {best_c}")
print(f"→ c teórico (√2):    1.414")


### Experimento 2 — Variación del presupuesto de tiempo

**Pregunta:** ¿Más tiempo siempre mejora el rendimiento?
La hipótesis es que existe un punto de saturación donde los rollouts aleatorios
agregan poco valor adicional — ese punto es el cuello de botella principal del agente.


In [ ]:
TIME_BUDGETS = [50, 100, 200, 500, 900]
C_OPTIMO     = best_c   # usamos el c óptimo del Experimento 1

results_exp2 = []

for budget in TIME_BUDGETS:
    print(f"  {budget}ms ...", end=' ')
    agent = PabloMCTS(time_budget_ms=budget, c=C_OPTIMO)
    rand  = RandomAgent()
    df    = play_match(agent, rand, n_games=N_GAMES_EXP,
                       time_budget_ms=budget, c=C_OPTIMO)
    df    = clean_df(df)

    per_game = df.groupby('game_id')['result_num'].first()
    win_rate  = (per_game == 1).mean()
    avg_sims  = df['simulations'].mean()
    avg_depth = df['tree_depth_max'].mean()

    results_exp2.append({
        'budget_ms': budget,
        'win_rate': win_rate,
        'avg_simulations': avg_sims,
        'avg_depth': avg_depth,
    })
    print(f"win rate = {win_rate:.0%}, sims = {avg_sims:.0f}")

df_exp2 = pd.DataFrame(results_exp2)
df_exp2


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Win rate vs tiempo
axes[0].plot(df_exp2['budget_ms'], df_exp2['win_rate'],
             marker='o', color='steelblue', linewidth=2)
axes[0].set_xlabel('Presupuesto de tiempo (ms/jugada)')
axes[0].set_ylabel('Win rate vs jugador aleatorio')
axes[0].set_title('Win rate vs presupuesto de tiempo')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Simulaciones vs tiempo
axes[1].plot(df_exp2['budget_ms'], df_exp2['avg_simulations'],
             marker='s', color='darkorange', linewidth=2)
axes[1].set_xlabel('Presupuesto de tiempo (ms/jugada)')
axes[1].set_ylabel('Simulaciones promedio por jugada')
axes[1].set_title('Simulaciones vs presupuesto de tiempo')

# Profundidad vs tiempo
axes[2].plot(df_exp2['budget_ms'], df_exp2['avg_depth'],
             marker='^', color='seagreen', linewidth=2)
axes[2].set_xlabel('Presupuesto de tiempo (ms/jugada)')
axes[2].set_ylabel('Profundidad máxima promedio')
axes[2].set_title('Profundidad del árbol vs presupuesto de tiempo')

plt.suptitle(f'Experimento 2 — Variación del presupuesto de tiempo | c={C_OPTIMO} | {N_GAMES_EXP} partidas')
plt.tight_layout()
plt.savefig('fig_exp2_time_budget.png', bbox_inches='tight')
plt.show()


### Experimento 3 — Self-play (agente vs sí mismo)

**Pregunta:** ¿Hay ventaja de color? ¿Qué pasa cuando dos instancias del mismo agente se enfrentan?
La rúbrica exige explícitamente este análisis.


In [ ]:
BUDGET_SELFPLAY = int(df_exp2.loc[df_exp2['win_rate'].idxmax(), 'budget_ms'])
print(f"Usando presupuesto del Experimento 2 con mayor win rate: {BUDGET_SELFPLAY}ms")

results_exp3 = []

for game_i in range(N_GAMES_EXP):
    agent_red    = PabloMCTS(time_budget_ms=BUDGET_SELFPLAY, c=C_OPTIMO)
    agent_yellow = PabloMCTS(time_budget_ms=BUDGET_SELFPLAY, c=C_OPTIMO)
    agent_red.mount()
    agent_yellow.mount()

    state   = ConnectState()
    players = {-1: agent_red, 1: agent_yellow}

    while not state.is_final():
        action = players[state.player].act(state.board)
        state  = state.transition(int(action))

    winner = state.get_winner()
    results_exp3.append({
        'game_id': game_i,
        'winner_color': 'Rojo' if winner == -1 else ('Amarillo' if winner == 1 else 'Empate'),
        'winner_num': winner,
    })

df_exp3 = pd.DataFrame(results_exp3)
counts = df_exp3['winner_color'].value_counts()
print("\nResultados self-play:")
print(counts)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribución de ganadores
color_map = {'Rojo': 'tomato', 'Amarillo': 'gold', 'Empate': 'lightgray'}
axes[0].bar(counts.index, counts.values,
            color=[color_map.get(c, 'steelblue') for c in counts.index])
axes[0].set_title('Distribución de ganadores en self-play')
axes[0].set_ylabel('Número de partidas')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 0.5, f'{val/N_GAMES_EXP:.0%}', ha='center', fontsize=11)

# Win rate acumulada a lo largo de las partidas
df_exp3['red_wins_cum']    = (df_exp3['winner_num'] == -1).cumsum() / (df_exp3.index + 1)
df_exp3['yellow_wins_cum'] = (df_exp3['winner_num'] ==  1).cumsum() / (df_exp3.index + 1)
axes[1].plot(df_exp3.index, df_exp3['red_wins_cum'],
             color='tomato', label='Rojo', linewidth=2)
axes[1].plot(df_exp3.index, df_exp3['yellow_wins_cum'],
             color='goldenrod', label='Amarillo', linewidth=2)
axes[1].axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='50%')
axes[1].set_title('Win rate acumulada — self-play')
axes[1].set_xlabel('Número de partidas')
axes[1].set_ylabel('Win rate acumulada')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].legend()

plt.suptitle(f'Experimento 3 — Self-play | c={C_OPTIMO} | {N_GAMES_EXP} partidas | {BUDGET_SELFPLAY}ms/jugada')
plt.tight_layout()
plt.savefig('fig_exp3_selfplay.png', bbox_inches='tight')
plt.show()


### Experimento 4 — Win rate por color vs aleatorio

**Pregunta:** ¿El agente gana en ambos colores? (requisito explícito de la rúbrica al 50%)


In [ ]:
results_exp4 = []

for color, color_label in [(-1, 'Rojo'), (1, 'Amarillo')]:
    print(f"  Corriendo como {color_label}...", end=' ')
    wins, draws, losses = 0, 0, 0

    for game_i in range(N_GAMES_EXP):
        agent = PabloMCTS(time_budget_ms=BUDGET_SELFPLAY, c=C_OPTIMO)
        rand  = RandomAgent()
        agent.mount()
        rand.mount()

        state = ConnectState()
        # Forzar color del agente
        if color == -1:
            players = {-1: agent, 1: rand}
        else:
            players = {-1: rand, 1: agent}

        while not state.is_final():
            action = players[state.player].act(state.board)
            state  = state.transition(int(action))

        winner = state.get_winner()
        if winner == color:   wins   += 1
        elif winner == 0:     draws  += 1
        else:                 losses += 1

    results_exp4.append({
        'color': color_label,
        'wins': wins, 'draws': draws, 'losses': losses,
        'win_rate': wins / N_GAMES_EXP,
        'n_games': N_GAMES_EXP,
    })
    print(f"win rate = {wins/N_GAMES_EXP:.0%}")

df_exp4 = pd.DataFrame(results_exp4)
df_exp4


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x      = np.arange(2)
width  = 0.25
colors = ['tomato', 'gold', 'lightgray']
labels = ['Victorias', 'Empates', 'Derrotas']
keys   = ['wins', 'draws', 'losses']

for i, (key, label, col) in enumerate(zip(keys, labels, colors)):
    bars = ax.bar(x + i * width, df_exp4[key], width, label=label, color=col, edgecolor='gray')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
                f'{h/N_GAMES_EXP:.0%}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(df_exp4['color'])
ax.set_ylabel('Número de partidas')
ax.set_title(f'Experimento 4 — Win rate por color vs aleatorio | {N_GAMES_EXP} partidas')
ax.axhline(N_GAMES_EXP * 0.5, linestyle='--', color='gray', alpha=0.5, label='50% (mínimo rúbrica)')
ax.legend()

plt.tight_layout()
plt.savefig('fig_exp4_by_color.png', bbox_inches='tight')
plt.show()


## 5. Evaluation — Analítica Predictiva con Scikit-learn

Entrenamos un clasificador para predecir el resultado de una partida
a partir de las variables internas del agente. El objetivo no es solo predecir,
sino **identificar qué variable interna del agente predice mejor una victoria**.
Ese insight guía directamente la propuesta de mejora.


In [ ]:
# Consolidar datos de todos los experimentos vs aleatorio
dfs_all = []

for c_val in C_VALUES:
    agent = PabloMCTS(time_budget_ms=200, c=c_val)
    rand  = RandomAgent()
    df    = play_match(agent, rand, n_games=50, time_budget_ms=200, c=c_val)
    df    = clean_df(df)
    dfs_all.append(df)

df_model = pd.concat(dfs_all, ignore_index=True)
print(f"Dataset para modelado: {df_model.shape}")
df_model[['simulations', 'tree_depth_max', 'root_confidence',
          'efficiency', 'c_param', 'game_result']].describe().round(3)


In [ ]:
# Features y target
FEATURES = ['simulations', 'tree_depth_max', 'root_confidence',
            'efficiency', 'c_param', 'my_color']

# Target: ganó (1) o no ganó (0) — clasificación binaria
df_model['target'] = (df_model['game_result'] == 'win').astype(int)

X = df_model[FEATURES].fillna(0)
y = df_model['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Balance de clases — Victorias: {y.mean():.0%} | No victorias: {1-y.mean():.0%}")


In [ ]:
# Entrenar Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['No victoria', 'Victoria']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Importancia de features
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Importancia de variables (Random Forest)')
axes[0].set_xlabel('Importancia relativa')

# Matriz de confusión
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf,
    display_labels=['No victoria', 'Victoria'],
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title('Matriz de confusión — Random Forest')

plt.tight_layout()
plt.savefig('fig_exp5_sklearn.png', bbox_inches='tight')
plt.show()

top_feature = importances.idxmax()
print(f"\n→ Variable más predictiva de victoria: '{top_feature}'")
print(f"   Importancia: {importances[top_feature]:.3f}")


## 6. Deployment — Conclusiones y Propuesta de Mejora

### Configuración óptima para el torneo


In [ ]:
print("=== Configuración óptima encontrada ===")
print(f"  c (UCB1):            {C_OPTIMO}")
print(f"  Presupuesto tiempo:  {BUDGET_SELFPLAY}ms/jugada")
print()

print("=== Resumen de resultados ===")
for _, row in df_exp4.iterrows():
    print(f"  Color {row['color']:8s}: {row['win_rate']:.0%} win rate vs aleatorio "
          f"({'✓ cumple' if row['win_rate'] > 0.5 else '✗ no cumple'} requisito mínimo)")
print()

red_wins    = df_exp3['winner_num'].eq(-1).mean()
yellow_wins = df_exp3['winner_num'].eq(1).mean()
print(f"  Self-play — Rojo gana: {red_wins:.0%} | Amarillo gana: {yellow_wins:.0%}")
print()
print(f"  Variable interna más predictiva de victoria: '{top_feature}'")


### Cuellos de botella identificados

**Bottleneck 1 — Saturación de rollouts aleatorios** *(más importante)*
El Experimento 2 muestra que a partir de cierto presupuesto de tiempo el win rate
deja de crecer aunque se acumulen más simulaciones. La causa es que los rollouts
completamente aleatorios tienen alta varianza: simulaciones adicionales no agregan
información útil porque el juego aleatorio no representa juego real.

**Mejora propuesta:** implementar una *rollout policy* con heurística mínima de tablero
(priorizar jugadas que completan 3 en raya propias o bloquean 3 en raya rivales)
en lugar de selección uniforme aleatoria. Esto reduciría la varianza por simulación
y desplazaría el punto de saturación hacia más tiempo, mejorando el win rate con el
mismo presupuesto.

**Bottleneck 2 — Profundidad del árbol**
El Experimento 2 también muestra que la profundidad máxima del árbol crece
lentamente con el tiempo: el árbol se expande más en anchura que en profundidad.
En Connect-4, la profundidad importa porque las amenazas de victoria suelen
requerir anticipación de 3-4 jugadas.

**Mejora propuesta:** implementar *progressive widening* para limitar la
expansión en anchura y forzar más profundidad con el mismo presupuesto de simulaciones.
